# GeoNUSAF Baseline — UNetFormer (ResNet-18 + global-local attention decoder)**Kathmandu Valley satellite land-use segmentation — 6 classes + ignore**One of **five** models in the final lineup: U-Net (ResNet-18), **UNetFormer (this)**, SegFormer-B0,SegNeXt-T, TC-SegFormer (proposed). Protocol consistency outranks peak accuracy: this notebookmirrors `unet-r18-regularized.ipynb` section for section, and only the model block differs.## Shared protocol — identical across all five notebooks| Item | Value ||---|---|| Split | 3-fold CV over **408 base tiles** (flip variants travel as one group) || Split modes | `block` (export-sequence proxy) **and** `random` (leakage-gap control) || Loss | effective-number-weighted CE (auto beta, capped, label smoothing 0.05) + Lovasz-Softmax, lambda 0.75 || Optimizer | AdamW, differential LR (encoder = decoder/10), weight decay **0.01** || Schedule | 500-step linear warmup, then cosine to zero over **`COS_HORIZON_EPOCHS = 120`** || Max epochs | 200 (the cosine finishes at 120; the tail is a flat-zero LR safety margin) || Early stop | val mIoU of the **EMA** weights, patience 25, **`MIN_DELTA = 0.002`** || Regularization | EMA 0.999, label smoothing 0.05, drop path 0.1, dropout 0.1, wd 0.01, grad clip 1.0 || Augmentation | rot90 (k in 0..3) + mild photometric jitter, genuinely **on-the-fly** || Seed | single seed = 42 || Ignore | index 255, excluded from loss and all metrics |## What is model-specific here- **UNetFormer**: timm ResNet-18 encoder (ImageNet) + global-local attention transformer decoder,  ~11.8 M params — same encoder as the U-Net baseline, so the two isolate the **decoder** difference.- **Auxiliary head at weight 0.4** on the 1/32 decoder feature (the paper's deep supervision).  Architectural, not a protocol change; `AUX_WEIGHT = 0.0` removes it.- **Independent implementation.** The reference repo (`WangLibo1995/GeoSeg`) is **GPL-3.0**, so copying  its `UNetFormer.py` would make your code release copyleft. Section 12 is written from the ISPRS  paper's architecture description instead. Cite the paper either way.## Nine changes from the earlier `unetformer_final.ipynb` — read before running1. **Cosine horizon is now 120 epochs, not 200.** Previously `total_steps = steps_per_epoch *   MAX_EPOCHS`, so with early stopping around epoch 70-90 the LR never annealed — it was still near   65-75% of peak when the run ended. `COS_HORIZON_EPOCHS` now controls decay independently of   `MAX_EPOCHS`, matching `segformer-b0-new.ipynb`.2. **EMA (0.999) added**, and validation / checkpointing / early stopping all run on the EMA weights.   This was the single biggest regularization gap against the other four models.3. **Label smoothing 0.05** on the CE term.4. **Drop path 0.1** on the three GLTB residual branches (the transformer-standard regularizer;   `DROP_PATH = 0.0` recovers the paper's plain blocks). Adds no `state_dict` keys.5. **`MIN_DELTA = 0.002`** with a separate reference tracker, so a +0.0003 flicker no longer resets   patience. `best.pt` still saves on any improvement.6. **`persistent_workers=False` on the train loader.** The old value froze augmentation at epoch 0 —   `train_ds.epoch = epoch` never reached the forked workers, so every image got one fixed rot90 /   jitter draw for the entire run. That was static pre-augmentation, not on-the-fly augmentation.7. **Determinism off by default.** `use_deterministic_algorithms(True)` forces a sort-based   `indexing_backward_kernel` for bilinear-upsample backward, which profiling attributed ~63% of all   CUDA time to. This model has five `F.interpolate` calls. `DETERMINISTIC = True` restores it.8. **`weight_decay = 0.01`** (was 1e-4), matching SegFormer-B0. `WEIGHT_DECAY_PAPER = 2.5e-4` is kept   in CFG for the record but unused.9. **`LR_DECODER = 3e-4`** (was 6e-4). See section 12 for why, and report the deviation.Plus two bug fixes: section 19 now pulls `best.pt` from Hugging Face rather than assuming it existson local disk (a resumed session that early-stopped without improving used to crash there), and`best_metrics` is restored on resume so the model card can never emit NaNs.## Session strategy (12 h Kaggle cap)Each **fold is one session**. Full state (model, EMA, optimizer, scaler, scheduler, patiencecounters) is pushed to Hugging Face as `last.pt`; on restart the notebook resumes from itautomatically, after verifying the architecture signature matches. Set `FOLD` and `SPLIT_MODE` inthe config cell, run top to bottom, repeat.## Prerequisites- **Internet ON** (Settings -> Internet). Needed for pip, ImageNet weights, W&B, HF.- Kaggle Secrets: `HF_TOKEN` (write scope) and `WANDB_API_KEY`.- GPU: T4 x1.

## 1. Environment`ptflops` and `einops` are not preinstalled on Kaggle. Everything else (torch, timm, cv2, skimage,sklearn, huggingface_hub, wandb) is. `ptflops` is pinned to the same version the other baselines usedso the GFLOPs column is measured identically; `einops` is used for the window partition in theattention block.

In [ ]:
!pip install -q ptflops==0.7.3 einops 2>&1 | tail -2import os, re, gc, json, time, math, copy, random, hashlib, warnings, itertoolsfrom dataclasses import dataclass, asdict, fieldfrom pathlib import Pathfrom typing import List, Dict, Tuple, Optionalimport numpy as npimport cv2import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoaderwarnings.filterwarnings("ignore")cv2.setNumThreads(0)   # critical: cv2 threads fight DataLoader workers on Kaggle's 4 vCPUsprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 2. Configuration**Set `FOLD` and `SPLIT_MODE` here, then run the whole notebook.** One fold per session.`SPLIT_MODE`:- `"block"` — contiguous runs of export-sequence order as a spatial-block proxy (the reported protocol)- `"random"` — fully random base-tile split (the leakage-gap control)Both group flip variants, so neither has *duplicate* leakage; the difference between them isolates*spatial* leakage specifically.The `REGULARIZATION` and `SCHEDULE` blocks below are byte-identical to the other four notebooks. Ifyou change a value in one, change it in all five and re-run all five — that is the whole point ofhaving a fixed protocol.

In [ ]:
@dataclassclass CFG:    # ---- run selection (the ONLY two fields you edit per session) ----    SPLIT_MODE: str = "block"          # "block" | "random"    FOLD: int = 0                      # 0 | 1 | 2    # ---- data ----    DATA_ROOT: str = "/kaggle/input/datasets/sugam24/satellite-datasets/satellite_dataset"    CACHE_DIR: str = "/kaggle/working/cache512"    PREBUILT_CACHE: Optional[str] = None   # point at a saved cache dataset to skip the repack    IMG_SIZE: int = 512    N_CLASSES: int = 6    IGNORE_INDEX: int = 255    PRELOAD_RAM: bool = True           # ~1.3 GB for the whole 512 cache; kills all disk I/O    # ---- split ----    N_FOLDS: int = 3    BLOCK_BUFFER: int = 2              # base tiles dropped from train at each block boundary    # ---- model (UNetFormer-specific) ----    ARCH_VERSION: str = "unetformer-r18-v2"   # stamped into every checkpoint; guards resume    ENCODER: str = "resnet18"          # timm, features_only    ENCODER_WEIGHTS: bool = True       # ImageNet pretrained (needs internet on first run)    DECODE_CHANNELS: int = 64          # paper default for the ResNet-18 variant    WINDOW_SIZE: int = 8    NUM_HEADS: int = 8    AUX_WEIGHT: float = 0.4            # deep supervision on the 1/32 feature; 0.0 disables    # ---- SHARED REGULARIZATION BLOCK (identical in all five notebooks) ----    DROPOUT: float = 0.10              # seg-head dropout    DROP_PATH: float = 0.10            # stochastic depth on the GLTB residual branches    LABEL_SMOOTHING: float = 0.05    WEIGHT_DECAY: float = 0.01    EMA_DECAY: float = 0.999           # 0.0 disables EMA    CLIP_NORM: float = 1.0    LAMBDA_LOVASZ: float = 0.75    EN_BETA: float = 0.9999            # effective-number beta (Cui et al. 2019)    WEIGHT_CAP: float = 10.0           # cap on normalized class weights    # ---- SHARED SCHEDULE BLOCK (identical in all five notebooks) ----    MAX_EPOCHS: int = 200              # hard ceiling    COS_HORIZON_EPOCHS: int = 120      # cosine decays to zero here, independent of MAX_EPOCHS    WARMUP_STEPS: int = 500    PATIENCE: int = 25    MIN_DELTA: float = 0.002           # improvement below this does not reset patience    # ---- optimization ----    BATCH_SIZE: int = 16    GRAD_ACCUM: int = 1                # raise to 2 and halve BATCH_SIZE if you OOM    LR_DECODER: float = 3e-4    LR_ENCODER: float = 3e-5           # 10x differential, same as every other baseline    LR_PAPER: float = 6e-4             # UNetFormer paper value -- NOT used; see section 12    WEIGHT_DECAY_PAPER: float = 2.5e-4 # not used -- kept for the record    # ---- augmentation (identical across all five baselines) ----    AUG_ROT90: bool = True    AUG_PHOTOMETRIC: bool = True    AUG_BRIGHTNESS: float = 0.10    AUG_CONTRAST: float = 0.10    # ---- runtime ----    SEED: int = 42    NUM_WORKERS: int = 4    AMP: bool = True    DETERMINISTIC: bool = False        # True costs ~6x on this architecture; see section 3    # ---- persistence ----    HF_USER: str = "sugam24"           # <-- set to your HF username    HF_PREFIX: str = "geonusaf-unetformer-r18"    PUSH_LAST_EVERY: int = 10          # epochs between full-state pushes    RESUME: bool = True                # False = ignore any last.pt and start clean    WANDB_PROJECT: str = "geonusaf-baselines"    OUT_DIR: str = "/kaggle/working/out"    CLASS_NAMES: tuple = ("Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural")cfg = CFG()os.makedirs(cfg.OUT_DIR, exist_ok=True)MODEL_TAG = "UNetFormer-R18"RUN_NAME = f"{cfg.HF_PREFIX}-{cfg.SPLIT_MODE}-fold{cfg.FOLD}"HF_REPO  = f"{cfg.HF_USER}/{RUN_NAME}"print("RUN:", RUN_NAME, "| HF repo:", HF_REPO)print(f"LR dec/enc {cfg.LR_DECODER:g}/{cfg.LR_ENCODER:g} | wd {cfg.WEIGHT_DECAY:g} | "      f"EMA {cfg.EMA_DECAY} | ls {cfg.LABEL_SMOOTHING} | dp {cfg.DROP_PATH} | aux {cfg.AUX_WEIGHT}")print(f"cosine horizon {cfg.COS_HORIZON_EPOCHS} ep | max {cfg.MAX_EPOCHS} ep | "      f"patience {cfg.PATIENCE} @ min_delta {cfg.MIN_DELTA}")

## 3. Seeding and determinism`DETERMINISTIC = False` is the default and it is a deliberate protocol decision, not sloppiness.`torch.use_deterministic_algorithms(True)` forces PyTorch to pick a sort-based`indexing_backward_kernel` for the backward pass of bilinear upsampling. Profiling on the T4attributed roughly **63% of total CUDA time** to that one kernel, and this decoder calls`F.interpolate` five times per forward. Measured effect on this architecture: full forward+backwardstep drops from ~2.65 s to ~0.45 s. Over three folds that is days of quota.Everything else stays pinned — Python / NumPy / torch RNG, worker seeding, the DataLoader generator,and the augmentation RNG, which is keyed on `(seed, epoch, index)` rather than on global state. Sosplits, fold membership, class weights and augmentation draws are all exactly reproducible; onlynon-deterministic GPU reduction order varies. State that caveat in the paper rather than claimingbit-exact reproducibility, and set `DETERMINISTIC = True` if a reviewer asks for a bit-exact rerun.

In [ ]:
def set_seed(seed: int, deterministic: bool):    random.seed(seed); np.random.seed(seed)    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)    os.environ["PYTHONHASHSEED"] = str(seed)    if deterministic:        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"        torch.backends.cudnn.deterministic = True        torch.backends.cudnn.benchmark = False        torch.use_deterministic_algorithms(True, warn_only=True)    else:        torch.backends.cudnn.deterministic = False        torch.backends.cudnn.benchmark = True          # fixed 512x512 shapes -> autotune pays off        torch.use_deterministic_algorithms(False)def worker_init_fn(worker_id):    s = cfg.SEED + worker_id    np.random.seed(s); random.seed(s)set_seed(cfg.SEED, cfg.DETERMINISTIC)g = torch.Generator(); g.manual_seed(cfg.SEED)print(f"seeded {cfg.SEED} | deterministic={cfg.DETERMINISTIC} | cudnn.benchmark={torch.backends.cudnn.benchmark}")

## 4. Filename parsing and base-tile groupingConfirmed convention: `image{N}.png` / `image{N}hf.png` / `image{N}vf.png`, masks mirror with the`mask` prefix. `N` is ascending **export order with gaps** (mono-class tiles were dropped afterexport), and the three variants of a tile are *not* adjacent in a sorted listing — so grouping isregex-based on `N`, never positional.The assertions here are the guard rail: if a future dataset version renames anything, this cellfails loudly instead of silently splitting a tile across folds.

In [ ]:
IMG_DIR  = Path(cfg.DATA_ROOT) / "images"MASK_DIR = Path(cfg.DATA_ROOT) / "masks"PAT = re.compile(r"^image(\d+)(hf|vf)?\.png$")assert IMG_DIR.is_dir(), (    f"{IMG_DIR} not found. 'images' dirs under /kaggle/input: "    + ", ".join(str(p.parent) for p in sorted(Path('/kaggle/input').rglob('images'))[:10]))records = []          # (base_id:int, variant:str, image_name:str, mask_name:str)for f in sorted(os.listdir(IMG_DIR)):    m = PAT.match(f)    assert m, f"unparsed image filename: {f}"    base_id, variant = int(m.group(1)), (m.group(2) or "orig")    mask_name = "mask" + f[len("image"):]    assert (MASK_DIR / mask_name).exists(), f"missing mask for {f}"    records.append((base_id, variant, f, mask_name))groups: Dict[int, List[tuple]] = {}for r in records:    groups.setdefault(r[0], []).append(r)BASE_IDS = sorted(groups)                      # ascending export order = the spatial proxy axisassert len(records) == 1224, len(records)assert len(BASE_IDS) == 408, len(BASE_IDS)assert all(len(v) == 3 for v in groups.values())gaps = [b - a for a, b in zip(BASE_IDS, BASE_IDS[1:]) if b - a > 1]print(f"{len(records)} files | {len(BASE_IDS)} base tiles | id range {BASE_IDS[0]}..{BASE_IDS[-1]}")print(f"{len(gaps)} gaps in export sequence (dropped tiles), total {sum(g-1 for g in gaps)} missing ids")

## 5. Mask colour diagnostic — run this before anything elseThe masks are RGB PNGs; the class table is in **BGR**. `cv2.imread` returns BGR, so the keys applydirectly — but note that `(128,0,0)` = Residential and `(0,0,128)` = River are exact reverses of eachother. **Getting the channel order wrong silently swaps your dominant class with your rarest one**and every metric downstream would still look plausible.The check below dumps every distinct colour actually present, then asserts that Residential occupiesfar more pixels than River. It also surfaces stray colours (CVAT polygon edge artifacts, resamplingresidue) — anything not in the 6-colour table becomes `ignore_index = 255`, so nothing outside`{0..5, 255}` ever reaches the loss.

In [ ]:
BGR_TO_CLASS = {    (128, 0, 0): 0,    # Residential Area    (0, 128, 0): 1,    # Road    (0, 0, 128): 2,    # River    (0, 128, 128): 3,  # Forest    (128, 128, 0): 4,  # Unused Land    (128, 0, 128): 5,  # Agricultural Area}# Pack BGR into a single uint32 key so the mapping is one vectorized lookup, not 6 boolean passes.def pack_bgr(a: np.ndarray) -> np.ndarray:    a = a.astype(np.uint32)    return (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]LUT = np.full(1 << 24, 255, dtype=np.uint8)          # 16 MB; default = ignorefor bgr, idx in BGR_TO_CLASS.items():    LUT[pack_bgr(np.array(bgr, dtype=np.uint8).reshape(1, 1, 3))[0, 0]] = idxdef rgbmask_to_index(bgr_img: np.ndarray) -> np.ndarray:    return LUT[pack_bgr(bgr_img)]# ---- diagnostic over a random sample of raw full-res masks ----rng = np.random.RandomState(cfg.SEED)sample = [records[i] for i in rng.choice(len(records), 40, replace=False)]colour_hits, class_px = {}, np.zeros(7, dtype=np.int64)   # 7th bucket = ignorefor _, _, _, mname in sample:    m = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)   # BGR    cols, cnts = np.unique(m.reshape(-1, 3), axis=0, return_counts=True)    for c, n in zip(cols, cnts):        colour_hits[tuple(int(x) for x in c)] = colour_hits.get(tuple(int(x) for x in c), 0) + int(n)    idx = rgbmask_to_index(m)    b = np.bincount(np.where(idx == 255, 6, idx).ravel(), minlength=7)    class_px += bknown = set(BGR_TO_CLASS)print("distinct BGR colours found in sample:")for c, n in sorted(colour_hits.items(), key=lambda kv: -kv[1]):    tag = f"-> class {BGR_TO_CLASS[c]} ({cfg.CLASS_NAMES[BGR_TO_CLASS[c]]})" if c in known else "-> IGNORE"    print(f"  BGR{c!s:<16} {n/sum(colour_hits.values())*100:6.2f}%  {tag}")tot = class_px.sum()print("\nclass pixel share (sample):")for i, nm in enumerate(list(cfg.CLASS_NAMES) + ["<ignore>"]):    print(f"  {i if i < 6 else 255:>3} {nm:<14} {class_px[i]/tot*100:6.2f}%")# Orientation guard: Residential must dominate River. If this trips, your channel order is flipped.assert class_px[0] > class_px[2] * 3, (    "Residential <= River -- BGR/RGB channel order is probably inverted. STOP and fix before training.")print("\nchannel-order check PASSED")

## 6. One-time repack to a 512x512 cacheThe tiles are **2345x2351 on disk (~10 MB each)**. Decoding 5.5 MP PNGs and resizing them on Kaggle's4 vCPUs would leave the T4 idle for most of every epoch.The repack does the resize **once**:- **Images**: `INTER_AREA` (the correct kernel for 4.6x downsampling; bilinear aliases thin roads).- **Masks**: colour -> class index at **full resolution first**, then `INTER_NEAREST` to 512.  Resizing the RGB mask first would interpolate between class colours and manufacture phantom classes  at every polygon edge.Result: ~700 MB instead of 13 GB. Effective GSD after this step is 300 m / 512 px = **0.586 m/px**,which is the figure to report; the native export is ~0.128 m/px.**The cache is model-independent** — build it once and set `PREBUILT_CACHE` in all five notebooks.

In [ ]:
from concurrent.futures import ProcessPoolExecutorCACHE = Path(cfg.PREBUILT_CACHE) if cfg.PREBUILT_CACHE else Path(cfg.CACHE_DIR)MARKER = CACHE / "_complete.json"def _repack_one(rec):    _, _, iname, mname = rec    img = cv2.imread(str(IMG_DIR / iname), cv2.IMREAD_COLOR)    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)    msk = cv2.imread(str(MASK_DIR / mname), cv2.IMREAD_COLOR)    lab = LUT[pack_bgr(msk)]                                    # full-res -> class index    lab = cv2.resize(lab, (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_NEAREST)    cv2.imwrite(str(CACHE / "images" / iname), img, [cv2.IMWRITE_PNG_COMPRESSION, 3])    cv2.imwrite(str(CACHE / "labels" / mname), lab, [cv2.IMWRITE_PNG_COMPRESSION, 3])    return np.bincount(np.where(lab == 255, 6, lab).ravel(), minlength=7)if MARKER.exists():    meta = json.load(open(MARKER))    print(f"cache present at {CACHE} ({meta['n']} pairs) -- skipping repack")else:    assert not cfg.PREBUILT_CACHE, f"PREBUILT_CACHE set but {MARKER} missing"    (CACHE / "images").mkdir(parents=True, exist_ok=True)    (CACHE / "labels").mkdir(parents=True, exist_ok=True)    t0 = time.time(); counts = {}    with ProcessPoolExecutor(max_workers=4) as ex:        for rec, b in zip(records, ex.map(_repack_one, records, chunksize=8)):            counts[rec[2]] = b.tolist()            if len(counts) % 200 == 0:                el = time.time() - t0                print(f"  {len(counts)}/{len(records)}  {el:.0f}s  eta {el/len(counts)*(len(records)-len(counts)):.0f}s")    json.dump({"n": len(counts), "counts": counts}, open(MARKER, "w"))    print(f"repacked {len(counts)} pairs in {time.time()-t0:.0f}s")PIX_COUNTS = {k: np.array(v, dtype=np.int64) for k, v in json.load(open(MARKER))["counts"].items()}sz = sum(f.stat().st_size for f in CACHE.rglob('*.png')) / 1e9print(f"cache size: {sz:.2f} GB")

## 7. Load the cache into RAM1224 x (512x512x3 + 512x512) uint8 = **~1.3 GB**, comfortably inside Kaggle's ~13 GB. Holding itresident removes disk I/O from the training loop entirely and lets `num_workers` do nothing butaugmentation. Set `PRELOAD_RAM = False` if you ever hit memory pressure.Note that workers are no longer persistent (section 10), so they re-fork each epoch. Fork iscopy-on-write and these are large contiguous arrays with few Python objects, so the cost isnegligible — but it is the reason `PRELOAD_RAM` matters more here than it did before.

In [ ]:
RAM = {}if cfg.PRELOAD_RAM:    t0 = time.time()    for i, (_, _, iname, mname) in enumerate(records):        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        RAM[iname] = (img, lab)        if (i + 1) % 400 == 0: print(f"  loaded {i+1}/{len(records)}")    mb = sum(a.nbytes + b.nbytes for a, b in RAM.values()) / 1e6    print(f"preloaded {len(RAM)} pairs, {mb:.0f} MB, {time.time()-t0:.0f}s")    # sanity: labels must only ever contain 0..5 or 255    u = np.unique(np.concatenate([b[::32, ::32].ravel() for _, b in RAM.values()]))    assert set(u.tolist()) <= set(range(6)) | {255}, f"unexpected label values: {u}"    print("label value check PASSED:", u.tolist())

## 8. Splitters — sequence-block proxy and random control### The proxy, and its honest limitationThere are no surviving grid indices, so contiguous runs of export order stand in for spatial blocks.QGIS Atlas iterates the coverage layer in feature order, which for a rectangular grid is normally**row-major**. That means:- Consecutive ids are neighbours **along the scan axis only**. Adjacency *perpendicular* to it (the  tile directly above/below) is uncontrolled — those tiles can land in different folds.- Every row wrap is a spatial discontinuity, so a "block" is really a set of strips.- Dropped mono-class tiles break runs, so id distance overstates ground distance near gaps.Net effect: **this proxy under-estimates true spatial leakage** relative to real geo-blocked CV.Report it as *sequence-block CV (export-order proxy)*, not as spatial CV, and cite the random-splitgap as a lower bound on the leakage effect.### Buffer`BLOCK_BUFFER = 2` base tiles adjacent to each held-out block are dropped from **train only** (neverfrom val), so the fold boundary is not two neighbouring tiles sitting on opposite sides of the split.Costs ~1% of training data.### GroupingBoth splitters operate on the 408 base ids; `expand()` then pulls in all three variants. A base tileand its flips can never straddle a fold. The splitters are byte-identical across all five notebooks,so fold membership matches exactly and every model is compared on the same held-out tiles.

In [ ]:
def block_split(base_ids: List[int], fold: int, n_folds: int, buffer: int):    n = len(base_ids)    edges = [round(i * n / n_folds) for i in range(n_folds + 1)]    lo, hi = edges[fold], edges[fold + 1]    val = base_ids[lo:hi]    train = base_ids[:max(0, lo - buffer)] + base_ids[min(n, hi + buffer):]    return train, valdef random_split(base_ids: List[int], fold: int, n_folds: int, seed: int):    ids = list(base_ids)    random.Random(seed).shuffle(ids)                 # same permutation for all folds    edges = [round(i * len(ids) / n_folds) for i in range(n_folds + 1)]    val = ids[edges[fold]:edges[fold + 1]]    train = ids[:edges[fold]] + ids[edges[fold + 1]:]    return sorted(train), sorted(val)if cfg.SPLIT_MODE == "block":    tr_ids, va_ids = block_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.BLOCK_BUFFER)elif cfg.SPLIT_MODE == "random":    tr_ids, va_ids = random_split(BASE_IDS, cfg.FOLD, cfg.N_FOLDS, cfg.SEED)else:    raise ValueError(cfg.SPLIT_MODE)def expand(ids): return [r for i in ids for r in groups[i]]tr_recs, va_recs = expand(tr_ids), expand(va_ids)assert not (set(tr_ids) & set(va_ids)), "base-tile leakage between train and val"assert not ({r[2] for r in tr_recs} & {r[2] for r in va_recs}), "file-level leakage between train and val"print(f"{cfg.SPLIT_MODE} fold {cfg.FOLD}: {len(tr_ids)} train tiles / {len(va_ids)} val tiles "      f"({len(BASE_IDS)-len(tr_ids)-len(va_ids)} buffered out)")print(f"  -> {len(tr_recs)} train images / {len(va_recs)} val images")print(f"  val id range: {min(va_ids)}..{max(va_ids)}")

## 9. Effective-number class weights — computed on the train split onlyWeights come from the **training fold alone**. Computing them once over the full dataset would leakval class priors into the loss; small effect, but exactly the kind of thing a Q1 reviewer flags.### Important: beta must scale with the counting unitEffective number (Cui et al. 2019) is `E_c = (1 - beta^n_c) / (1 - beta)`, `w_c = 1/E_c`. The paperapplies it to **per-class image counts** (hundreds to thousands), where `beta = 0.9999` is sensiblebecause `1/(1-beta) = 1e4` is the same order as `n_c`.Here `n_c` is a **pixel count** (~1e6 to 1e9). At that scale `beta^n_c` underflows to 0 for *every*class, so `E_c = 1/(1-beta)` for all c and the weights come out **exactly uniform** — i.e. plainunweighted CE, silently. The cell auto-selects `beta = 1 - 1/N_total` instead, which puts thesaturation point at the actual data scale and recovers a meaningful spread before capping.`EN_BETA_AUTO = True` **matches all four other notebooks**. This is the one switch that must neverdiffer between them.

In [ ]:
EN_BETA_AUTO = True     # False = reproduce the legacy (saturated -> uniform) behaviourdef effective_number_weights(recs, cap, n_classes, beta_fixed, auto):    n = np.zeros(n_classes, dtype=np.float64)    for _, _, iname, _ in recs:        n += PIX_COUNTS[iname][:n_classes]    beta = (1.0 - 1.0 / n.sum()) if auto else beta_fixed    eff = (1.0 - np.power(beta, n)) / (1.0 - beta)    w = 1.0 / np.maximum(eff, 1e-12)    w = w / w.mean()    w = np.minimum(w, cap)    return w / w.mean(), n, betaCLASS_W, TRAIN_PX, BETA_USED = effective_number_weights(    tr_recs, cfg.WEIGHT_CAP, cfg.N_CLASSES, cfg.EN_BETA, EN_BETA_AUTO)print(f"total train pixels: {TRAIN_PX.sum():.3e} | beta used: {BETA_USED:.12f} "      f"| saturation point 1/(1-beta) = {1/(1-BETA_USED):.3e}")print(f"{'class':<14}{'train px %':>12}{'weight':>10}")for i, nm in enumerate(cfg.CLASS_NAMES):    print(f"{nm:<14}{TRAIN_PX[i]/TRAIN_PX.sum()*100:11.2f}%{CLASS_W[i]:10.3f}")print(f"weight ratio max/min: {CLASS_W.max()/CLASS_W.min():.1f}")if CLASS_W.max() / CLASS_W.min() < 1.05:    print("\nWARNING: weights are effectively uniform -- beta has saturated. This is unweighted CE.")CLASS_W_T = torch.tensor(CLASS_W, dtype=torch.float32).cuda()

## 10. Dataset and DataLoaders**No hflip/vflip** — those are already materialised in the 1224 files. Adding them would produceduplicate views inside a single epoch.On-the-fly augmentation is **rot90 (k in 0..3) + mild brightness/contrast jitter**. Combined with thebaked-in flips this completes the dihedral group D4, the natural symmetry group for nadir overheadimagery, and it introduces no new interpolation (rot90 is exact, so 10-20 px roads stay crisp). Scalejitter is deliberately excluded: it changes effective GSD, which would make the thin-structureresults harder to interpret.Augmentation is seeded per `(seed, epoch, index)`, so the draw is reproducible, workers nevercollide, and **the draw genuinely changes every epoch**.> **Changed from the previous version.** `persistent_workers` is now `False` on the train loader.> With it set to `True`, the worker processes hold their own fork of `train_ds`, so the> `train_ds.epoch = epoch` assignment in section 17 never reached them — `self.epoch` stayed 0 for> the whole run and each index got *one fixed* rot90/jitter draw. That is static pre-augmentation,> not on-the-fly augmentation, and it removed most of the regularization value of this cell. The val> loader keeps `persistent_workers=True` because it does no augmentation and never reads `epoch`.

In [ ]:
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)class SatSeg(Dataset):    def __init__(self, recs, train: bool):        self.recs, self.train, self.epoch = recs, train, 0    def __len__(self): return len(self.recs)    def _read(self, iname, mname):        if cfg.PRELOAD_RAM:            img, lab = RAM[iname]            return img.copy(), lab.copy()        img = cv2.imread(str(CACHE / "images" / iname), cv2.IMREAD_COLOR)        lab = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        return img, lab    def __getitem__(self, i):        _, _, iname, mname = self.recs[i]        img, lab = self._read(iname, mname)        if self.train:            rs = np.random.RandomState((cfg.SEED * 100003 + self.epoch * 9973 + i) % (2**31))            if cfg.AUG_ROT90:                k = rs.randint(4)                if k:                    img = np.ascontiguousarray(np.rot90(img, k))                    lab = np.ascontiguousarray(np.rot90(lab, k))            if cfg.AUG_PHOTOMETRIC:                a = 1.0 + rs.uniform(-cfg.AUG_CONTRAST, cfg.AUG_CONTRAST)        # contrast                b = 255.0 * rs.uniform(-cfg.AUG_BRIGHTNESS, cfg.AUG_BRIGHTNESS)  # brightness                img = np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)        x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0        x = (x - IMNET_MEAN) / IMNET_STD        x = torch.from_numpy(x.transpose(2, 0, 1))        y = torch.from_numpy(lab.astype(np.int64))        return x, ytrain_ds, val_ds = SatSeg(tr_recs, True), SatSeg(va_recs, False)train_ld = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, drop_last=True,                      num_workers=cfg.NUM_WORKERS, pin_memory=True,                      persistent_workers=False,      # MUST stay False: workers must see the new epoch                      worker_init_fn=worker_init_fn, generator=g)val_ld   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,                      num_workers=cfg.NUM_WORKERS, pin_memory=True, persistent_workers=True,                      worker_init_fn=worker_init_fn)# Proof that augmentation actually varies by epoch -- this is the check the old notebook lacked.train_ds.epoch = 0; a0, _ = train_ds[0]train_ds.epoch = 1; a1, _ = train_ds[0]train_ds.epoch = 0print("augmentation varies across epochs:", bool((a0 - a1).abs().max() > 1e-6))xb, yb = next(iter(train_ld))print(xb.shape, xb.dtype, yb.shape, yb.dtype, "| label values:", torch.unique(yb).tolist()[:8])

## 11. Loss — weighted CE (label-smoothed) + Lovasz-SoftmaxLovasz-Softmax is the convex surrogate of the Jaccard loss (Berman et al. 2018) and is what actuallymoves mIoU on thin classes; weighted CE keeps early optimization stable. `lambda = 0.75`, constant —no ramp, so the reported training loss is comparable across every epoch and across models.Three implementation details that matter:- **`label_smoothing = 0.05`** on the CE term (new; matches SegFormer-B0 and the U-Net). With 408  tiles and coarse polygon annotations, hard targets at polygon edges are frequently wrong, and  smoothing is the cheapest defence against memorising them.- The whole loss is computed in **fp32 outside autocast**. Lovasz sorts per-pixel errors and its  gradient is a cumulative-sum difference; in fp16 that accumulates visible error and can produce NaNs.- Ignore pixels are filtered *before* flattening, so 255 never reaches either term.UNetFormer's auxiliary head reuses the same `ComboLoss` object rather than getting a second lossdefinition — the aux term is applied in the training loop at `AUX_WEIGHT`, which keeps the lossfunction itself identical across all five baselines.

In [ ]:
def lovasz_grad(gt_sorted):    p = len(gt_sorted); gts = gt_sorted.sum()    inter = gts - gt_sorted.float().cumsum(0)    union = gts + (1 - gt_sorted).float().cumsum(0)    jac = 1.0 - inter / union    if p > 1: jac[1:p] = jac[1:p] - jac[0:-1]    return jacdef lovasz_softmax_flat(probas, labels, n_classes):    if probas.numel() == 0: return probas.sum() * 0.0    losses = []    for c in range(n_classes):        fg = (labels == c).float()        if fg.sum() == 0: continue                     # "present" classes only        err = (fg - probas[:, c]).abs()        err_sorted, perm = torch.sort(err, 0, descending=True)        losses.append(torch.dot(err_sorted, lovasz_grad(fg[perm])))    return torch.stack(losses).mean() if losses else probas.sum() * 0.0class ComboLoss(nn.Module):    def __init__(self, weights, lam, ignore, n_classes, label_smoothing):        super().__init__()        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=ignore,                                      label_smoothing=label_smoothing)        self.lam, self.ignore, self.nc = lam, ignore, n_classes    def forward(self, logits, target):        logits = logits.float()                        # fp32 regardless of autocast        ce = self.ce(logits, target)        B, C, H, W = logits.shape        p = logits.permute(0, 2, 3, 1).reshape(-1, C).softmax(1)        t = target.reshape(-1)        keep = t != self.ignore        lv = lovasz_softmax_flat(p[keep], t[keep], self.nc)        return ce + self.lam * lv, ce.detach(), lv.detach()criterion = ComboLoss(CLASS_W_T, cfg.LAMBDA_LOVASZ, cfg.IGNORE_INDEX,                      cfg.N_CLASSES, cfg.LABEL_SMOOTHING).cuda()print(f"loss ready | lambda {cfg.LAMBDA_LOVASZ} | label smoothing {cfg.LABEL_SMOOTHING} "      f"| aux weight {cfg.AUX_WEIGHT}")

## 12. Model, optimizer, EMA, schedulerWhere the U-Net baseline calls `smp.Unet(...)`, this notebook carries the architecture itself: thereference implementation (`WangLibo1995/GeoSeg`) is **GPL-3.0**, and copying its`geoseg/models/UNetFormer.py` would put your paper's code release under GPL-3.0 too. What follows iswritten from the ISPRS paper's description (Wang et al. 2022, *ISPRS J. Photogramm. Remote Sens.*190:196-214) rather than copied. Cite the paper for the method either way.| Stage | Resolution (512 in) | Component ||---|---|---|| Encoder | /4 /8 /16 /32 | timm ResNet-18, ImageNet, 64/128/256/512 ch || /32 | 16x16 | 1x1 to 64 ch -> Global-Local Transformer Block || /16, /8 | 32x32, 64x64 | Weighted Fusion with skip -> GLTB || /4 | 128x128 | Feature Refinement Head (pixel + channel attention) || Head | -> 512x512 | 3x3 conv, dropout, 1x1 to 6 logits, bilinear upsample |The GLTB runs windowed multi-head self-attention (global branch, window 8) in parallel with a convbranch (local branch), sums them, adds a cross-shaped average-pooled context term, and projects witha separable conv. All decoder feature maps are 16/32/64/128 px, every one divisible by the windowsize, so no window padding ever fires at 512x512.### Learning rate — read this before writing the paperThe UNetFormer paper's published setting is `6e-4`. **This notebook uses `3e-4`**, and that is adeliberate deviation with a stated reason: the paper trains on UAVid / Vaihingen / Potsdam, which areone to two orders of magnitude larger than 408 base tiles, and at `6e-4` this is the only model inthe lineup running at 10x the LR of the other four *and* the only one whose regularization stack wasthinner. Holding the LR at a published value while every other axis is matched would confound anyarchitecture conclusion you draw from the comparison table.Report it as: *the published `6e-4` was reduced to `3e-4` for parity with the other baselines giventhe dataset scale; `LR_PAPER` is retained in the config.* If the first ten epochs show grad-norm maxin single digits and healthy mIoU movement, you can also run one fold at `6e-4` as a supplementarysensitivity row.### Regularization added here`DROP_PATH = 0.1` applies stochastic depth to the attention and MLP residual branches of the threeGLTBs — the transformer-standard regularizer, and the direct analogue of SegFormer-B0's `DROP_PATH`.It adds **no `state_dict` keys**, so it is invisible at inference and does not break checkpointcompatibility. `DROP_PATH = 0.0` recovers the paper's plain blocks.### Two structural choices, unchangedThe aux head is built unconditionally rather than gated on `self.training` in `__init__` as thereference does — that gating changes `state_dict` keys depending on the mode the model happened to beconstructed in, which breaks resume and standalone inference. And weight decay is applied uniformlyacross all parameters, exactly as in the other four notebooks; excluding norm/bias/positional-biasterms is a common transformer refinement, but it is not what the others did, so it stays out.### EMAAn exponential moving average of the weights at decay 0.999, with the standard warmup ramp`min(decay, (1+step)/(10+step))` so the average is usable from the first epochs rather than beingpinned to the initialization. **Validation, checkpointing and early stopping all read the EMAweights**, and `best.pt` stores them — so the model you publish is the averaged one, matchingSegFormer-B0 and the U-Net. Buffers (BatchNorm running statistics) are averaged too, which mattershere because this decoder is BatchNorm-heavy.### Schedule500 linear warmup steps, then cosine to zero over `COS_HORIZON_EPOCHS = 120`. `MAX_EPOCHS = 200` isonly a ceiling — past epoch 120 the LR is clamped at zero, so training there is a no-op and earlystopping will have fired long before. **This is the change that matters most**: previously the cosinewas spread over 200 epochs, so a run that stopped at epoch 80 ended with the LR still near 70% ofpeak and never annealed at all.

In [ ]:
import timmfrom einops import rearrange# ---------------------------------------------------------------- building blocksclass Conv(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1, bias=False):        super().__init__(nn.Conv2d(i, o, k, stride=stride, dilation=dilation,                                   padding=((stride - 1) + dilation * (k - 1)) // 2, bias=bias))class ConvBN(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, o, k, stride=stride, dilation=dilation,                                   padding=((stride - 1) + dilation * (k - 1)) // 2, bias=False),                         nn.BatchNorm2d(o))class ConvBNReLU(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, o, k, stride=stride, dilation=dilation,                                   padding=((stride - 1) + dilation * (k - 1)) // 2, bias=False),                         nn.BatchNorm2d(o), nn.ReLU6(inplace=True))class SeparableConvBN(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, i, k, stride=stride, dilation=dilation,                                   padding=((stride - 1) + dilation * (k - 1)) // 2, groups=i, bias=False),                         nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o))class DropPath(nn.Module):    # Stochastic depth on a residual branch. No parameters, no state_dict keys.    def __init__(self, p=0.0):        super().__init__(); self.p = float(p)    def forward(self, x):        if self.p == 0.0 or not self.training: return x        keep = 1.0 - self.p        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)        return x * mask / keepclass Mlp(nn.Module):    def __init__(self, dim, hidden, drop=0.0):        super().__init__()        self.fc1, self.act = nn.Conv2d(dim, hidden, 1), nn.ReLU6(inplace=True)        self.fc2, self.drop = nn.Conv2d(hidden, dim, 1), nn.Dropout(drop)    def forward(self, x):        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))# ---------------------------------------------------------------- global-local attentionclass GlobalLocalAttention(nn.Module):    # Windowed MHSA (global branch) in parallel with a conv branch (local), plus pooled cross context.    def __init__(self, dim=64, num_heads=8, window_size=8, qkv_bias=False):        super().__init__()        assert dim % num_heads == 0        self.nh, self.ws = num_heads, window_size        self.scale = (dim // num_heads) ** -0.5        self.qkv = Conv(dim, 3 * dim, k=1, bias=qkv_bias)        self.local1, self.local2 = ConvBN(dim, dim, k=3), ConvBN(dim, dim, k=1)        self.proj = SeparableConvBN(dim, dim, k=window_size)        # kernel (ws,1) with padding ws//2-1 shrinks that axis by exactly 1; the pad before proj restores it        self.attn_x = nn.AvgPool2d((window_size, 1), stride=1, padding=(window_size // 2 - 1, 0))        self.attn_y = nn.AvgPool2d((1, window_size), stride=1, padding=(0, window_size // 2 - 1))        ws = window_size        self.rpb_table = nn.Parameter(torch.zeros((2 * ws - 1) ** 2, num_heads))        nn.init.trunc_normal_(self.rpb_table, std=0.02)        c = torch.stack(torch.meshgrid(torch.arange(ws), torch.arange(ws), indexing="ij")).flatten(1)        rel = (c[:, :, None] - c[:, None, :]).permute(1, 2, 0).contiguous()        rel[:, :, 0] += ws - 1; rel[:, :, 1] += ws - 1; rel[:, :, 0] *= 2 * ws - 1        self.register_buffer("rpi", rel.sum(-1), persistent=False)    def forward(self, x):        B, C, H, W = x.shape        local = self.local2(x) + self.local1(x)        ws = self.ws        ph, pw = (ws - H % ws) % ws, (ws - W % ws) % ws        xp = F.pad(x, (0, pw, 0, ph), mode="reflect") if (ph or pw) else x        Hp, Wp = xp.shape[-2:]        q, k, v = rearrange(self.qkv(xp),                            "b (qkv h d) (hh w1) (ww w2) -> qkv (b hh ww) h (w1 w2) d",                            qkv=3, h=self.nh, w1=ws, w2=ws)        bias = self.rpb_table[self.rpi.view(-1)].view(ws * ws, ws * ws, self.nh).permute(2, 0, 1)        attn = ((q @ k.transpose(-2, -1)) * self.scale + bias.unsqueeze(0)).softmax(-1) @ v        attn = rearrange(attn, "(b hh ww) h (w1 w2) d -> b (h d) (hh w1) (ww w2)",                         b=B, hh=Hp // ws, ww=Wp // ws, w1=ws, w2=ws)[:, :, :H, :W]        fused = attn + local        cross = (self.attn_x(F.pad(fused, (0, 0, 0, 1), mode="reflect")) +                 self.attn_y(F.pad(fused, (0, 1, 0, 0), mode="reflect")))        return self.proj(F.pad(fused + cross, (0, 1, 0, 1), mode="reflect"))[:, :, :H, :W]class Block(nn.Module):    # Global-Local Transformer Block. BatchNorm rather than LayerNorm, as in the paper.    def __init__(self, dim=64, num_heads=8, window_size=8, mlp_ratio=4.0, drop=0.0, drop_path=0.0):        super().__init__()        self.norm1, self.norm2 = nn.BatchNorm2d(dim), nn.BatchNorm2d(dim)        self.attn = GlobalLocalAttention(dim, num_heads, window_size)        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)        self.dp1, self.dp2 = DropPath(drop_path), DropPath(drop_path)    def forward(self, x):        x = x + self.dp1(self.attn(self.norm1(x)))        return x + self.dp2(self.mlp(self.norm2(x)))# ---------------------------------------------------------------- fusion / headsclass WF(nn.Module):    # Weighted fusion of an upsampled decoder feature with an encoder skip; the weights are learned.    def __init__(self, in_ch, dc, eps=1e-8):        super().__init__()        self.pre_conv, self.post_conv = Conv(in_ch, dc, k=1), ConvBNReLU(dc, dc, k=3)        self.weights, self.eps = nn.Parameter(torch.ones(2, dtype=torch.float32)), eps    def forward(self, x, res):        x = F.interpolate(x, size=res.shape[-2:], mode="bilinear", align_corners=False)        w = torch.relu(self.weights); w = w / (w.sum() + self.eps)        return self.post_conv(w[0] * self.pre_conv(res) + w[1] * x)class FeatureRefinementHead(nn.Module):    def __init__(self, in_ch, dc, eps=1e-8):        super().__init__()        self.pre_conv, self.post_conv = Conv(in_ch, dc, k=1), ConvBNReLU(dc, dc, k=3)        self.weights, self.eps = nn.Parameter(torch.ones(2, dtype=torch.float32)), eps        self.pa = nn.Sequential(nn.Conv2d(dc, dc, 3, padding=1, groups=dc), nn.Sigmoid())        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), Conv(dc, max(dc // 16, 4), k=1), nn.ReLU6(True),                                Conv(max(dc // 16, 4), dc, k=1), nn.Sigmoid())        self.shortcut, self.proj = ConvBN(dc, dc, k=1), SeparableConvBN(dc, dc, k=3)        self.act = nn.ReLU6(True)    def forward(self, x, res):        x = F.interpolate(x, size=res.shape[-2:], mode="bilinear", align_corners=False)        w = torch.relu(self.weights); w = w / (w.sum() + self.eps)        x = self.post_conv(w[0] * self.pre_conv(res) + w[1] * x)        sc = self.shortcut(x)        x = self.pa(x) * x + self.ca(x) * x        return self.act(self.proj(x) + sc)class AuxHead(nn.Module):    def __init__(self, dc, n_cls):        super().__init__()        self.body = nn.Sequential(ConvBNReLU(dc, dc), nn.Dropout2d(0.1), Conv(dc, n_cls, k=1))    def forward(self, x): return self.body(x)class UNetFormer(nn.Module):    def __init__(self, encoder="resnet18", n_classes=6, dc=64, window_size=8,                 num_heads=8, dropout=0.1, drop_path=0.1, pretrained=True):        super().__init__()        self.backbone = timm.create_model(encoder, features_only=True,                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)        ec = self.backbone.feature_info.channels()          # (64, 128, 256, 512) at /4 /8 /16 /32        self.pre_conv = ConvBN(ec[3], dc, k=1)        self.b4 = Block(dc, num_heads, window_size, drop_path=drop_path)        self.p3, self.b3 = WF(ec[2], dc), Block(dc, num_heads, window_size, drop_path=drop_path)        self.p2, self.b2 = WF(ec[1], dc), Block(dc, num_heads, window_size, drop_path=drop_path)        self.p1 = FeatureRefinementHead(ec[0], dc)        self.aux_head = AuxHead(dc, n_classes)              # always built: keeps state_dict keys stable        self.seg_head = nn.Sequential(ConvBNReLU(dc, dc), nn.Dropout2d(dropout), Conv(dc, n_classes, k=1))    def forward(self, x):        h, w = x.shape[-2:]        r1, r2, r3, r4 = self.backbone(x)        d = self.b4(self.pre_conv(r4))        aux = self.aux_head(d)        d = self.b3(self.p3(d, r3))        d = self.b2(self.p2(d, r2))        d = self.p1(d, r1)        out = F.interpolate(self.seg_head(d), (h, w), mode="bilinear", align_corners=False)        if self.training:            return out, F.interpolate(aux, (h, w), mode="bilinear", align_corners=False)        return out# ---------------------------------------------------------------- EMAclass ModelEMA:    # Exponential moving average of parameters AND buffers, with a warmup ramp on the decay.    def __init__(self, model, decay):        self.decay, self.updates = decay, 0        self.module = copy.deepcopy(model).eval()        for p in self.module.parameters(): p.requires_grad_(False)    @torch.no_grad()    def update(self, model):        self.updates += 1        d = min(self.decay, (1 + self.updates) / (10 + self.updates))        msd = model.state_dict()        for k, v in self.module.state_dict().items():            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)            else:                         v.copy_(msd[k])    def state_dict(self): return {"module": self.module.state_dict(), "updates": self.updates}    def load_state_dict(self, sd):        self.module.load_state_dict(sd["module"]); self.updates = sd["updates"]# ---------------------------------------------------------------- builddef build_model(pretrained=None):    return UNetFormer(        encoder=cfg.ENCODER,        n_classes=cfg.N_CLASSES,        dc=cfg.DECODE_CHANNELS,        window_size=cfg.WINDOW_SIZE,        num_heads=cfg.NUM_HEADS,        dropout=cfg.DROPOUT,        drop_path=cfg.DROP_PATH,        pretrained=cfg.ENCODER_WEIGHTS if pretrained is None else pretrained,    )model = build_model().cuda()# Shape smoke test: crash in section 12, not at hour four. Train -> (out, aux), eval -> out.model.train()with torch.no_grad():    _o, _a = model(torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).cuda())assert _o.shape == (2, cfg.N_CLASSES, cfg.IMG_SIZE, cfg.IMG_SIZE), _o.shapeassert _a.shape == _o.shape, _a.shapemodel.eval()with torch.no_grad():    _e = model(torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE).cuda())assert torch.is_tensor(_e) and _e.shape[1] == cfg.N_CLASSESdel _o, _a, _e; torch.cuda.empty_cache()# Architecture signature: keys + shapes. Stamped into every checkpoint and verified before any load,# so a checkpoint from an older architecture can never be silently or half-loaded.def arch_signature(m):    s = "|".join(f"{k}:{tuple(v.shape)}" for k, v in sorted(m.state_dict().items()))    return hashlib.md5(s.encode()).hexdigest()[:12]ARCH_SIG = arch_signature(model)ema = ModelEMA(model, cfg.EMA_DECAY) if cfg.EMA_DECAY > 0 else Noneenc_p = list(model.backbone.parameters())enc_ids = {id(p) for p in enc_p}dec_p = [p for p in model.parameters() if id(p) not in enc_ids]optimizer = torch.optim.AdamW(    [{"params": enc_p, "lr": cfg.LR_ENCODER, "name": "encoder"},     {"params": dec_p, "lr": cfg.LR_DECODER, "name": "decoder"}],    weight_decay=cfg.WEIGHT_DECAY)steps_per_epoch = len(train_ld) // cfg.GRAD_ACCUMcos_total_steps = steps_per_epoch * cfg.COS_HORIZON_EPOCHS      # NOT MAX_EPOCHSdef lr_lambda(step):    if step < cfg.WARMUP_STEPS:        return (step + 1) / cfg.WARMUP_STEPS    prog = (step - cfg.WARMUP_STEPS) / max(1, cos_total_steps - cfg.WARMUP_STEPS)    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))     # clamped: flat zero past the horizonscheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)scaler = torch.amp.GradScaler("cuda", enabled=cfg.AMP)n_par = sum(p.numel() for p in model.parameters())print(f"params: {n_par/1e6:.2f} M ({sum(p.numel() for p in enc_p)/1e6:.2f} M encoder / "      f"{sum(p.numel() for p in dec_p)/1e6:.2f} M decoder)")print(f"arch signature: {ARCH_SIG} ({cfg.ARCH_VERSION}) | EMA: {'on' if ema else 'off'}")print(f"steps/epoch {steps_per_epoch} | cosine over {cfg.COS_HORIZON_EPOCHS} ep "      f"= {cos_total_steps} steps | warmup {cfg.WARMUP_STEPS} steps")print(f"LR at ep 0/30/60/90/120: " + " / ".join(    f"{lr_lambda(e*steps_per_epoch)*cfg.LR_DECODER:.2e}" for e in (0, 30, 60, 90, 120)))

## 13. Metrics moduleIdentical to the other four notebooks — same class, same function, same keys, so the result JSONs canbe concatenated without any reconciliation.Everything accumulates into a single 6x6 confusion matrix built on-GPU with `bincount`. Ignore pixelsare dropped before accumulation, so 255 never enters any metric.- **IoU / F1 / mIoU / mF1** — macro over the 6 classes- **OA**, **frequency-weighted IoU**- **Cohen's kappa** — from the same matrix- **Producer's accuracy** = recall (row-normalized), **User's accuracy** = precision (column-normalized).  These are the remote-sensing names for the same quantities; reporting both plus the normalized  matrix is what reviewers in this field expect.- **clDice** (Shit et al. 2021) for **Road** and **River** only — a topology-aware score that rewards  connectivity rather than area overlap, the right lens for 10-20 px linear features. Skeletonization  is CPU-bound, so it runs at final evaluation only, not every epoch.

In [ ]:
from skimage.morphology import skeletonizeclass ConfMat:    def __init__(self, n): self.n = n; self.m = torch.zeros(n, n, dtype=torch.int64, device="cuda")    def update(self, pred, tgt):        k = tgt != cfg.IGNORE_INDEX        if k.sum() == 0: return        self.m += torch.bincount(self.n * tgt[k].long() + pred[k].long(),                                 minlength=self.n ** 2).reshape(self.n, self.n)    def numpy(self): return self.m.cpu().numpy().astype(np.float64)def metrics_from_cm(cm, names):    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tot = cm.sum()    iou = tp / np.maximum(tp + fp + fn, 1e-12)    f1  = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)    prod = tp / np.maximum(cm.sum(1), 1e-12)      # recall    user = tp / np.maximum(cm.sum(0), 1e-12)      # precision    freq = cm.sum(1) / max(tot, 1e-12)    po = tp.sum() / max(tot, 1e-12)    pe = (cm.sum(0) * cm.sum(1)).sum() / max(tot ** 2, 1e-12)    return {        "mIoU": float(np.nanmean(iou)), "mF1": float(np.nanmean(f1)),        "OA": float(po), "FWIoU": float((freq * iou).sum()),        "kappa": float((po - pe) / max(1 - pe, 1e-12)),        "per_class": {n: {"IoU": float(iou[i]), "F1": float(f1[i]),                          "producer_acc": float(prod[i]), "user_acc": float(user[i]),                          "support_px": int(cm.sum(1)[i])} for i, n in enumerate(names)},        "confusion_matrix_raw": cm.astype(np.int64).tolist(),        "confusion_matrix_norm": (cm / np.maximum(cm.sum(1, keepdims=True), 1e-12)).tolist(),    }def cldice_pair(pred_bin, gt_bin):    if gt_bin.sum() == 0 and pred_bin.sum() == 0: return None    sp = skeletonize(pred_bin.astype(bool)); sg = skeletonize(gt_bin.astype(bool))    tprec = (sp & gt_bin.astype(bool)).sum() / max(sp.sum(), 1)     # topology precision    tsens = (sg & pred_bin.astype(bool)).sum() / max(sg.sum(), 1)   # topology sensitivity    if tprec + tsens == 0: return 0.0    return float(2 * tprec * tsens / (tprec + tsens))

## 14. Efficiency metricsParams, GFLOPs at 512x512, peak training VRAM, wall-clock, and inference FPS. FPS is measured atbatch 1 with 20 warmup iterations and CUDA synchronization — without the warmup you are timing cuDNNautotuning, and on a shared Kaggle T4 you should treat FPS as indicative rather than exact.`GFLOPs_512` stays on `ptflops` so the column is measured with the same code as the other four.**One caveat specific to this model:** ptflops counts via module hooks, and the windowed attentionhere is `q @ k.T` and `attn @ v` written as tensor ops, not modules — so those matmuls are invisibleto it and `GFLOPs_512` under-reports. A second field, `GFLOPs_512_flopcounter`, uses`torch.utils.flop_counter`, which traces at the dispatcher and does see them. Report the ptflopsnumber in the comparison column for consistency and give the traced number in the text.

In [ ]:
def efficiency_report(m):    from ptflops import get_model_complexity_info    m.eval()    with torch.amp.autocast("cuda", enabled=False):        macs, params = get_model_complexity_info(            m, (3, cfg.IMG_SIZE, cfg.IMG_SIZE),            as_strings=False, print_per_layer_stat=False, verbose=False)    x = torch.randn(1, 3, cfg.IMG_SIZE, cfg.IMG_SIZE, device="cuda")    # Cross-check: ptflops misses the attention matmuls (tensor ops, not modules).    try:        from torch.utils.flop_counter import FlopCounterMode        fc = FlopCounterMode(display=False)        with fc, torch.no_grad():            m(x)        traced_gflops = fc.get_total_flops() / 1e9    except Exception as e:        print("flop_counter unavailable:", type(e).__name__); traced_gflops = float("nan")    with torch.no_grad(), torch.amp.autocast("cuda", enabled=cfg.AMP):        for _ in range(20): m(x)        torch.cuda.synchronize(); t0 = time.time()        for _ in range(50): m(x)        torch.cuda.synchronize()    return {"params_M": params / 1e6,            "GFLOPs_512": macs * 2 / 1e9,           # ptflops returns MACs; FLOPs = 2 x MACs            "GFLOPs_512_flopcounter": traced_gflops,            "inference_fps_bs1": 50 / (time.time() - t0)}

## 15. Hugging Face persistence and resumeEach fold gets its **own repo** (`geonusaf-unetformer-r18-{mode}-fold{k}`) so any single fold can bepulled independently for inference later — nothing is ever overwritten across folds. Same layout asthe other four models, so one loader script works for all of them.Two artifacts per repo:- `best.pt` — **EMA weights** + config + metrics + arch signature, pushed whenever val mIoU improves.  This is what the inference script downloads.- `last.pt` — full state (raw weights, EMA, optimizer, scaler, scheduler, patience counters, history),  pushed every `PUSH_LAST_EVERY` epochs. This is what a new session resumes from.Both carry `arch_sig` and `arch_version`. Section 18 refuses to load a checkpoint whose signaturedoes not match the model built in section 12, and deletes the stale files from the Hub rather thanleaving a trap for the next session — this is the failure that cost a TC-SegFormer run.

In [ ]:
from huggingface_hub import HfApi, create_repo, hf_hub_downloadtry:    from kaggle_secrets import UserSecretsClient    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")except Exception as e:    HF_TOKEN = os.environ.get("HF_TOKEN"); print("secret fallback:", e)api = HfApi(token=HF_TOKEN)create_repo(HF_REPO, token=HF_TOKEN, exist_ok=True, private=False)def hf_push(local_path, name, msg):    api.upload_file(path_or_fileobj=local_path, path_in_repo=name,                    repo_id=HF_REPO, token=HF_TOKEN, commit_message=msg)def hf_try_download(name, repo=None):    try: return hf_hub_download(repo or HF_REPO, name, token=HF_TOKEN)    except Exception: return Nonedef hf_delete(name):    try:        api.delete_file(path_in_repo=name, repo_id=HF_REPO, token=HF_TOKEN,                        commit_message=f"remove stale {name} (architecture mismatch)")        print(f"  deleted stale {name} from {HF_REPO}")    except Exception as e:        print(f"  could not delete {name}: {type(e).__name__}")def push_model_card(best_metrics, epoch):    pc = best_metrics.get("per_class", {})    L = []    L.append("---")    L.append("tags: [semantic-segmentation, remote-sensing, unetformer, resnet18, geonusaf]")    L.append("library_name: pytorch")    L.append("---")    L.append("")    L.append(f"# GeoNUSAF - UNetFormer (ResNet-18) - {cfg.SPLIT_MODE} split, fold {cfg.FOLD}")    L.append("")    L.append("Kathmandu Valley land-use segmentation, 6 classes, ignore_index=255.")    L.append("")    L.append("| field | value |")    L.append("|---|---|")    L.append("| architecture | UNetFormer, timm resnet18 encoder (ImageNet), global-local attention decoder |")    L.append(f"| arch version | {cfg.ARCH_VERSION} (sig {ARCH_SIG}) |")    L.append(f"| split mode | {cfg.SPLIT_MODE} |")    L.append(f"| fold | {cfg.FOLD} of {cfg.N_FOLDS} |")    L.append(f"| seed | {cfg.SEED} |")    L.append("| input | 512x512, ImageNet norm, effective GSD 0.586 m/px |")    L.append(f"| classes | {', '.join(cfg.CLASS_NAMES)} |")    L.append(f"| lr (dec/enc) | {cfg.LR_DECODER:g} / {cfg.LR_ENCODER:g}, AdamW wd {cfg.WEIGHT_DECAY:g} |")    L.append(f"| schedule | {cfg.WARMUP_STEPS}-step warmup, cosine over {cfg.COS_HORIZON_EPOCHS} epochs |")    L.append(f"| regularization | EMA {cfg.EMA_DECAY}, label smoothing {cfg.LABEL_SMOOTHING}, "             f"drop path {cfg.DROP_PATH}, dropout {cfg.DROPOUT} |")    L.append(f"| aux head weight | {cfg.AUX_WEIGHT} |")    L.append(f"| weights | EMA |")    L.append(f"| best epoch | {epoch} |")    L.append(f"| val mIoU | {best_metrics.get('mIoU', float('nan')):.4f} |")    L.append(f"| val mF1 | {best_metrics.get('mF1', float('nan')):.4f} |")    L.append(f"| val OA | {best_metrics.get('OA', float('nan')):.4f} |")    L.append(f"| val kappa | {best_metrics.get('kappa', float('nan')):.4f} |")    L.append("")    L.append("## Per-class (validation)")    L.append("| class | IoU | F1 |")    L.append("|---|---|---|")    for k, v in pc.items():        L.append(f"| {k} | {v['IoU']:.4f} | {v['F1']:.4f} |")    L.append("")    L.append("Split caveat: `block` is sequence-block CV using an export-order proxy, not spatial CV.")    L.append("")    L.append("LR deviates from the UNetFormer paper's 6e-4 (reduced to 3e-4 for parity with the "             "other baselines at this dataset scale).")    L.append("")    L.append("Architecture is an independent implementation of Wang et al. (2022), "             "ISPRS J. Photogramm. Remote Sens. 190:196-214 (the reference repo is GPL-3.0).")    L.append("")    L.append("Checkpoint `best.pt` holds `model_state` (EMA weights) plus `cfg`, `metrics`, `arch_sig`.")    card = "\n".join(L)    p = f"{cfg.OUT_DIR}/README.md"; open(p, "w").write(card)    hf_push(p, "README.md", f"card epoch {epoch}")print("HF ready:", HF_REPO)

## 16. Weights & BiasesMatches the logging on the other four baselines. `resume="allow"` with a deterministic run id means aresumed session continues the same W&B run rather than starting a second one.

In [ ]:
import wandbtry:    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))    WANDB = Trueexcept Exception as e:    print("wandb disabled:", e); WANDB = Falseif WANDB:    wandb.init(project=cfg.WANDB_PROJECT, name=RUN_NAME, id=RUN_NAME,               resume="allow", config={**asdict(cfg), "arch_sig": ARCH_SIG, "model": MODEL_TAG})

## 17. Train / validate loopsMechanics worth knowing:- `scaler.unscale_(optimizer)` runs **before** `clip_grad_norm_`, otherwise you would be clipping  loss-scaled gradients and the 1.0 threshold would mean nothing. `clip_grad_norm_` returns the  pre-clip norm, which is what gets logged — that is the direct check against gradient explosion.- `train_ds.epoch = epoch` now actually reaches the workers, because they are no longer persistent.- **EMA is updated after every optimizer step**, and `validate()` takes the model to evaluate as an  argument. The main loop passes the EMA module, so every reported number, every checkpoint and the  early-stopping decision all refer to the averaged weights. The raw model is validated too, but only  as a logged diagnostic (`val_mIoU_raw`) — it is never what stopping or saving keys off.- Validation runs at full 512x512 with no TTA. Keep it that way across all baselines; TTA belongs in  a separate row of the table, not in the protocol.- Early stopping tracks **val mIoU**, not val loss — the loss is class-weighted and moves for reasons  mIoU does not care about.- In train mode UNetFormer returns `(out, aux)` and the aux logits get the same `ComboLoss` at  `AUX_WEIGHT`. The `isinstance` guard means `AUX_WEIGHT = 0.0` is a clean ablation rather than a  code-path change.

In [ ]:
def train_one_epoch(epoch):    model.train(); train_ds.epoch = epoch    tot, n, gn_acc = 0.0, 0, []    optimizer.zero_grad(set_to_none=True)    for it, (x, y) in enumerate(train_ld):        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.amp.autocast("cuda", enabled=cfg.AMP):            out = model(x)        logits, aux = out if isinstance(out, tuple) else (out, None)        loss, ce, lv = criterion(logits, y)                             # fp32, outside autocast        if aux is not None and cfg.AUX_WEIGHT > 0:            loss = loss + cfg.AUX_WEIGHT * criterion(aux, y)[0]         # deep supervision        scaler.scale(loss / cfg.GRAD_ACCUM).backward()        if (it + 1) % cfg.GRAD_ACCUM == 0:            scaler.unscale_(optimizer)                                  # unscale BEFORE clipping            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.CLIP_NORM)            gn_acc.append(float(gn))            scaler.step(optimizer); scaler.update()            optimizer.zero_grad(set_to_none=True); scheduler.step()            if ema is not None: ema.update(model)        tot += float(loss) * x.size(0); n += x.size(0)        if it % 20 == 0:            print(f"  e{epoch} it{it}/{len(train_ld)} loss {float(loss):.4f} "                  f"(ce {float(ce):.4f} lov {float(lv):.4f}) gnorm {gn_acc[-1] if gn_acc else 0:.3f}",                  flush=True)    return tot / max(n, 1), float(np.mean(gn_acc)), float(np.max(gn_acc))@torch.no_grad()def validate(net):    net.eval(); cm = ConfMat(cfg.N_CLASSES); tot, n = 0.0, 0    for x, y in val_ld:        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.amp.autocast("cuda", enabled=cfg.AMP):            logits = net(x)        loss, _, _ = criterion(logits, y)        cm.update(logits.float().argmax(1), y)        tot += float(loss) * x.size(0); n += x.size(0)    return tot / max(n, 1), metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)EVAL_NET = ema.module if ema is not None else model      # what every reported number refers toprint("validating on:", "EMA weights" if ema is not None else "raw weights")

## 18. Main loop with automatic resumeOn start this pulls `last.pt` from the fold's HF repo, verifies `arch_sig` and `arch_version` againstthe model built in section 12, and continues from that epoch. If the signature does not match — anolder architecture, a changed `DECODE_CHANNELS`, anything — it says so, deletes the stale files fromthe Hub, and starts clean rather than throwing a `load_state_dict` error four cells later.So the answer to "what if it exceeds 12 hours" is: it stops, you start a new session, run thenotebook top to bottom unchanged, and it picks up where it left off — optimizer, scaler, schedulerstep count, EMA, patience counters and history all included.### Early stopping, preciselyTwo counters, because they answer different questions:- `best_miou` / `best.pt` — updated on **any** improvement, however small. This is what you publish.- `ref_miou` / `bad` — updated only on an improvement of at least `MIN_DELTA = 0.002`. This is what  decides when to stop. Without it, a run that has plateaued keeps resetting its patience on +0.0003  noise and trains another 40 epochs past its useful life, deepening the train/val gap the whole way.### Expected cost~816 train images, batch 16, 51 steps/epoch. With determinism off and the 512 cache in RAM, expectroughly **25-45 s/epoch** including validation of both EMA and raw weights. The cosine finishes atepoch 120, and early stopping typically fires between 60 and 110, so budget **0.5-1.5 h per fold**.VRAM at batch 16 should peak around 7-10 GB of the T4's 16 GB (the EMA copy adds ~50 MB). The batchsize stays 16 regardless: the headroom is not an invitation to raise it, because effective batch ispart of the fixed protocol.

In [ ]:
start_epoch, best_miou, best_epoch, ref_miou, bad = 0, -1.0, -1, -1.0, 0history, best_metrics = [], {}ck = hf_try_download("last.pt") if cfg.RESUME else Noneif ck:    st = torch.load(ck, map_location="cuda", weights_only=False)    same_arch = (st.get("arch_sig") == ARCH_SIG and st.get("arch_version") == cfg.ARCH_VERSION)    same_fold = (st["cfg"]["SPLIT_MODE"] == cfg.SPLIT_MODE and st["cfg"]["FOLD"] == cfg.FOLD)    assert same_fold, "last.pt is from a different split/fold -- wrong repo for this config"    if not same_arch:        print(f"ARCHITECTURE MISMATCH: checkpoint {st.get('arch_version')} / {st.get('arch_sig')} "              f"vs current {cfg.ARCH_VERSION} / {ARCH_SIG}")        print("  -> discarding stale checkpoints and starting fresh")        hf_delete("last.pt"); hf_delete("best.pt")    else:        model.load_state_dict(st["model_state"])        if ema is not None and st.get("ema_state") is not None: ema.load_state_dict(st["ema_state"])        optimizer.load_state_dict(st["optim_state"])        scheduler.load_state_dict(st["sched_state"]); scaler.load_state_dict(st["scaler_state"])        start_epoch = st["epoch"] + 1        best_miou, best_epoch = st["best_miou"], st["best_epoch"]        ref_miou, bad = st["ref_miou"], st["bad"]        history = st["history"]; best_metrics = st.get("best_metrics", {})        print(f"RESUMED from epoch {st['epoch']} (best mIoU {best_miou:.4f} @ e{best_epoch}, bad={bad})")else:    print("fresh start")torch.cuda.reset_peak_memory_stats()t_start = time.time()for epoch in range(start_epoch, cfg.MAX_EPOCHS):    te = time.time()    tr_loss, gn_mean, gn_max = train_one_epoch(epoch)    va_loss, m = validate(EVAL_NET)                       # EMA -- the reported numbers    _, m_raw = validate(model)                            # raw -- diagnostic only    peak_gb = torch.cuda.max_memory_allocated() / 1e9    row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, "val_mIoU": m["mIoU"],           "val_mIoU_raw": m_raw["mIoU"], "val_mF1": m["mF1"], "val_OA": m["OA"],           "val_kappa": m["kappa"], "gap_train_val": va_loss - tr_loss,           "grad_norm_mean": gn_mean, "grad_norm_max": gn_max,           "lr_dec": scheduler.get_last_lr()[1], "peak_vram_gb": peak_gb,           "epoch_sec": time.time() - te, "bad_epochs": bad,           **{f"IoU/{k}": v["IoU"] for k, v in m["per_class"].items()}}    history.append(row)    if WANDB: wandb.log(row, step=epoch)    print(f"[e{epoch}] train {tr_loss:.4f} | val {va_loss:.4f} | mIoU(ema) {m['mIoU']:.4f} "          f"| mIoU(raw) {m_raw['mIoU']:.4f} | gnorm mu{gn_mean:.2f}/max{gn_max:.2f} "          f"| lr {row['lr_dec']:.2e} | {row['epoch_sec']:.0f}s | vram {peak_gb:.1f}GB")    # --- best checkpoint: any improvement at all ---    if m["mIoU"] > best_miou:        best_miou, best_epoch, best_metrics = m["mIoU"], epoch, m        p = f"{cfg.OUT_DIR}/best.pt"        torch.save({"model_state": EVAL_NET.state_dict(), "cfg": asdict(cfg),                    "metrics": m, "epoch": epoch, "arch_sig": ARCH_SIG,                    "arch_version": cfg.ARCH_VERSION, "weights": "ema" if ema else "raw"}, p)        hf_push(p, "best.pt", f"best mIoU {best_miou:.4f} @ e{epoch}")        push_model_card(best_metrics, epoch)        print(f"  -> new best {best_miou:.4f}, pushed")    # --- patience: only a MIN_DELTA improvement counts ---    if m["mIoU"] > ref_miou + cfg.MIN_DELTA:        ref_miou, bad = m["mIoU"], 0    else:        bad += 1    if (epoch % cfg.PUSH_LAST_EVERY == 0) or (bad >= cfg.PATIENCE) or (epoch == cfg.MAX_EPOCHS - 1):        p = f"{cfg.OUT_DIR}/last.pt"        torch.save({"model_state": model.state_dict(),                    "ema_state": ema.state_dict() if ema is not None else None,                    "optim_state": optimizer.state_dict(), "sched_state": scheduler.state_dict(),                    "scaler_state": scaler.state_dict(), "cfg": asdict(cfg),                    "arch_sig": ARCH_SIG, "arch_version": cfg.ARCH_VERSION,                    "epoch": epoch, "best_miou": best_miou, "best_epoch": best_epoch,                    "ref_miou": ref_miou, "bad": bad,                    "best_metrics": best_metrics, "history": history}, p)        hf_push(p, "last.pt", f"state e{epoch}")    if bad >= cfg.PATIENCE:        print(f"EARLY STOP at epoch {epoch}: {bad} epochs without a +{cfg.MIN_DELTA} improvement "              f"(best {best_miou:.4f} @ e{best_epoch})")        breakwall_h = (time.time() - t_start) / 3600print(f"training done | best mIoU {best_miou:.4f} @ epoch {best_epoch} | this session {wall_h:.2f} h")

## 19. Final evaluation of the best checkpointReloads `best.pt` — **from Hugging Face, falling back to local disk** — and computes the full metricsuite on the held-out fold, including clDice for Road and River (skeletonization per image, CPU) andthe efficiency block.The HF-first order is the fix for a real failure: in a resumed session that early-stopped withoutever improving, no `best.pt` was written to local disk this session, and the old code's`torch.load(f"{OUT_DIR}/best.pt")` raised `FileNotFoundError` after hours of training. The weights in`best.pt` are the EMA weights, so this evaluates exactly what the model card reports.clDice is averaged over images in which the class is present in GT or prediction; images where theclass is absent from both are skipped rather than scored as 1.0, which would inflate the number badlygiven how sparse River is.

In [ ]:
bp = hf_try_download("best.pt") or f"{cfg.OUT_DIR}/best.pt"ckb = torch.load(bp, map_location="cuda", weights_only=False)assert ckb.get("arch_sig") == ARCH_SIG, (    f"best.pt architecture {ckb.get('arch_sig')} != current {ARCH_SIG}")best_model = build_model(pretrained=False).cuda()best_model.load_state_dict(ckb["model_state"])best_model.eval()print(f"evaluating best.pt from epoch {ckb['epoch']} ({ckb.get('weights','raw')} weights), "      f"reported mIoU {ckb['metrics']['mIoU']:.4f}")cm = ConfMat(cfg.N_CLASSES)cld = {1: [], 2: []}                                   # Road, Riverwith torch.no_grad():    for x, y in val_ld:        x = x.cuda(non_blocking=True); y = y.cuda(non_blocking=True)        with torch.amp.autocast("cuda", enabled=cfg.AMP):            logits = best_model(x)        pred = logits.float().argmax(1)        cm.update(pred, y)        pn, yn = pred.cpu().numpy(), y.cpu().numpy()        valid = yn != cfg.IGNORE_INDEX        for b in range(pn.shape[0]):            for c in cld:                pb = (pn[b] == c) & valid[b]                gb = (yn[b] == c) & valid[b]                v = cldice_pair(pb, gb)                if v is not None: cld[c].append(v)final = metrics_from_cm(cm.numpy(), cfg.CLASS_NAMES)final["clDice"] = {cfg.CLASS_NAMES[c]: (float(np.mean(v)) if v else None) for c, v in cld.items()}final["clDice_n_images"] = {cfg.CLASS_NAMES[c]: len(v) for c, v in cld.items()}final.update(efficiency_report(best_model))final["peak_train_vram_gb"] = float(torch.cuda.max_memory_allocated() / 1e9)final["train_wall_clock_h_this_session"] = wall_hfinal["best_epoch"] = ckb["epoch"]final["stopped_epoch"] = history[-1]["epoch"] if history else Nonefinal["run"] = {"model": MODEL_TAG, "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD,                "seed": cfg.SEED, "arch_version": cfg.ARCH_VERSION, "arch_sig": ARCH_SIG,                "weights": ckb.get("weights", "raw"),                "cos_horizon_epochs": cfg.COS_HORIZON_EPOCHS,                "lr_decoder": cfg.LR_DECODER, "weight_decay": cfg.WEIGHT_DECAY,                "ema_decay": cfg.EMA_DECAY, "label_smoothing": cfg.LABEL_SMOOTHING,                "drop_path": cfg.DROP_PATH, "min_delta": cfg.MIN_DELTA}print(json.dumps({k: v for k, v in final.items()                  if k not in ("confusion_matrix_raw", "confusion_matrix_norm", "per_class")}, indent=2))print("\nper class:")for k, v in final["per_class"].items():    print(f"  {k:<14} IoU {v['IoU']:.4f}  F1 {v['F1']:.4f}  PA {v['producer_acc']:.4f}  UA {v['user_acc']:.4f}")

## 20. Persist resultsWrites `results_{mode}_fold{k}.json` (full, incl. confusion matrices), a flat `.csv` row for thecomparison table, and `history.csv` for the curves — then pushes all three to the fold's HF repo sothey survive the session. Build the five-model table later by pulling these from HF.The flat row carries the same column names as the other four notebooks, plus`GFLOPs_512_flopcounter`, so `pd.concat` across all five works and the extra column simply reads`NaN` for the CNN baselines that never needed it.

In [ ]:
import csvtag = f"{cfg.SPLIT_MODE}_fold{cfg.FOLD}"jp = f"{cfg.OUT_DIR}/results_{tag}.json"json.dump(final, open(jp, "w"), indent=2)flat = {"model": MODEL_TAG, "split_mode": cfg.SPLIT_MODE, "fold": cfg.FOLD, "seed": cfg.SEED,        "mIoU": final["mIoU"], "mF1": final["mF1"], "OA": final["OA"],        "FWIoU": final["FWIoU"], "kappa": final["kappa"],        "params_M": final["params_M"], "GFLOPs_512": final["GFLOPs_512"],        "GFLOPs_512_flopcounter": final["GFLOPs_512_flopcounter"],        "fps_bs1": final["inference_fps_bs1"], "peak_vram_gb": final["peak_train_vram_gb"],        "best_epoch": final["best_epoch"], "stopped_epoch": final["stopped_epoch"],        "cos_horizon": cfg.COS_HORIZON_EPOCHS, "lr_decoder": cfg.LR_DECODER,        "weight_decay": cfg.WEIGHT_DECAY, "ema": cfg.EMA_DECAY}for k, v in final["per_class"].items():    flat[f"IoU_{k}"] = v["IoU"]; flat[f"F1_{k}"] = v["F1"]for k, v in final["clDice"].items():    flat[f"clDice_{k}"] = vcp = f"{cfg.OUT_DIR}/results_{tag}.csv"with open(cp, "w", newline="") as f:    w = csv.DictWriter(f, fieldnames=list(flat)); w.writeheader(); w.writerow(flat)hp = f"{cfg.OUT_DIR}/history_{tag}.csv"keys = sorted({k for r in history for k in r})with open(hp, "w", newline="") as f:    w = csv.DictWriter(f, fieldnames=keys); w.writeheader()    for r in history: w.writerow({k: r.get(k) for k in keys})for p, n in [(jp, f"results_{tag}.json"), (cp, f"results_{tag}.csv"), (hp, f"history_{tag}.csv")]:    hf_push(p, n, "results")print("results pushed to", HF_REPO)

## 21. Overfitting curve — train vs val, and where the best epoch landedThis figure is the one to keep an eye on across all five models. Three things to read off it:1. **Where val loss bottoms versus where mIoU peaks.** They usually disagree — the loss is   class-weighted, so it starts climbing while mIoU is still improving. That divergence is normal and   is exactly why early stopping keys off mIoU.2. **The train/val gap trajectory.** Under the earlier configuration the U-Net drove train loss to   ~0.05 while val climbed to ~2.8. If this run shows the same shape, the regularization stack is   still not enough and the next lever is `DROP_PATH` up to 0.2 and `WEIGHT_DECAY` to 0.05.3. **EMA versus raw mIoU.** EMA should sit at or above raw and be visibly smoother. If EMA is   consistently *below* raw, the decay is too slow for the number of steps in this run — report it   and drop to 0.995.

In [ ]:
import matplotlib.pyplot as pltimport pandas as pdH = pd.DataFrame(history)fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))ax[0].plot(H.epoch, H.train_loss, label="train")ax[0].plot(H.epoch, H.val_loss, label="val")ax[0].axvline(final["best_epoch"], ls="--", c="k", lw=1, label=f"best e{final['best_epoch']}")ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("loss"); ax[0].legend(fontsize=8)ax[1].plot(H.epoch, H.val_mIoU, label="val mIoU (EMA)")if "val_mIoU_raw" in H: ax[1].plot(H.epoch, H.val_mIoU_raw, alpha=0.5, label="val mIoU (raw)")ax[1].axvline(final["best_epoch"], ls="--", c="k", lw=1)ax[1].set_xlabel("epoch"); ax[1].set_title("val mIoU"); ax[1].legend(fontsize=8)ax[2].plot(H.epoch, H.lr_dec, c="tab:green")ax[2].axvline(cfg.COS_HORIZON_EPOCHS, ls=":", c="r", lw=1, label=f"cosine horizon {cfg.COS_HORIZON_EPOCHS}")ax[2].set_xlabel("epoch"); ax[2].set_title("decoder LR"); ax[2].legend(fontsize=8)fig.suptitle(f"{MODEL_TAG} - {cfg.SPLIT_MODE} fold {cfg.FOLD}")plt.tight_layout()p = f"{cfg.OUT_DIR}/curves_{tag}.png"plt.savefig(p, dpi=140, bbox_inches="tight"); plt.show()hf_push(p, os.path.basename(p), "curves")lr_end = float(H.lr_dec.iloc[-1]) / cfg.LR_DECODERprint(f"LR at the final epoch was {lr_end*100:.1f}% of peak "      f"({'annealed as intended' if lr_end < 0.15 else 'STILL HIGH -- the run ended before the cosine did'})")print(f"train/val gap: start {H.gap_train_val.iloc[0]:+.3f} -> end {H.gap_train_val.iloc[-1]:+.3f}")

## 22. Aggregation across folds — run after all folds finishPulls each fold's CSV from its own HF repo and reports mean +/- std per split mode. The**block-mode mIoU minus random-mode mIoU is the leakage gap** — the headline number justifying whythe block protocol is the one you report. Expect random to score higher; if the gap is near zero,that is itself a finding worth stating (it would suggest the export-order proxy is not capturing muchspatial structure, consistent with the row-major limitation noted in section 8).

In [ ]:
rows = []for mode in ["block", "random"]:    for k in range(cfg.N_FOLDS):        repo = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-{mode}-fold{k}"        try:            rows.append(pd.read_csv(hf_hub_download(repo, f"results_{mode}_fold{k}.csv", token=HF_TOKEN)))        except Exception as e:            print(f"  missing {mode} fold {k}: {type(e).__name__}")if rows:    df = pd.concat(rows, ignore_index=True)    df.to_csv(f"{cfg.OUT_DIR}/aggregate_unetformer_r18.csv", index=False)    num = df.select_dtypes("number").columns    agg = df.groupby("split_mode")[num].agg(["mean", "std"])    print(agg[[c for c in [("mIoU","mean"),("mIoU","std"),("mF1","mean"),("kappa","mean")] if c in agg]])    if {"block", "random"} <= set(df.split_mode):        b = df[df.split_mode == "block"].mIoU.mean(); r = df[df.split_mode == "random"].mIoU.mean()        print(f"\nLEAKAGE GAP (random - block) mIoU: {r-b:+.4f}  [block {b:.4f} | random {r:.4f}]")

## 23. Qualitative triptychs — image / GT / predictionThe palette is the BGR table reversed to RGB for matplotlib; ignore renders black. The selection isdeliberately not random: it shows the highest- and lowest-mIoU validation tiles, because the failurecases are what a reviewer wants to see. Watch specifically for the two known confusions — Unused Landvs Agricultural (spectrally similar) and urban trees vs Forest — plus Road fragmentation, which iswhere clDice and IoU will disagree.For this model there is a third thing worth looking at: whether errors align with the 8x8 attentionwindow grid at the coarse stages. Blocky artefacts on that pitch would point at the window partitionrather than at the data.

In [ ]:
from matplotlib.patches import PatchPALETTE_RGB = np.zeros((256, 3), np.uint8)for bgr, idx in BGR_TO_CLASS.items():    PALETTE_RGB[idx] = bgr[::-1]                       # BGR table -> RGB for matplotlibPALETTE_RGB[255] = (0, 0, 0)def colorize(lab): return PALETTE_RGB[lab]REC_POS = {r[2]: i for i, r in enumerate(va_recs)}     # O(1) lookup instead of list.index@torch.no_grad()def per_image_miou(recs, k=3):    scores = []    for j in range(0, len(recs), 16):        chunk = recs[j:j + 16]        xs, ys = zip(*[val_ds[REC_POS[r[2]]] for r in chunk])        x = torch.stack(xs).cuda(); y = torch.stack(ys).cuda()        with torch.amp.autocast("cuda", enabled=cfg.AMP):            p = best_model(x).float().argmax(1)        for b, r in enumerate(chunk):            c = ConfMat(cfg.N_CLASSES); c.update(p[b], y[b])            scores.append((metrics_from_cm(c.numpy(), cfg.CLASS_NAMES)["mIoU"], r))    scores.sort(key=lambda t: t[0])    return scores[-k:][::-1], scores[:k]best_model.eval()best_imgs, worst_imgs = per_image_miou(va_recs, k=3)def triptych(items, title):    fig, axes = plt.subplots(len(items), 3, figsize=(13, 4.3 * len(items)))    axes = np.atleast_2d(axes)    for r, (score, rec) in enumerate(items):        _, _, iname, mname = rec        img = cv2.cvtColor(cv2.imread(str(CACHE / "images" / iname)), cv2.COLOR_BGR2RGB)        gt = cv2.imread(str(CACHE / "labels" / mname), cv2.IMREAD_GRAYSCALE)        x, _ = val_ds[REC_POS[iname]]        with torch.no_grad(), torch.amp.autocast("cuda", enabled=cfg.AMP):            pr = best_model(x[None].cuda()).float().argmax(1)[0].cpu().numpy().astype(np.uint8)        pr_vis = np.where(gt == 255, 255, pr).astype(np.uint8)   # blank ignore for fair viewing        for c, (im, t) in enumerate([(img, iname), (colorize(gt), "ground truth"),                                     (colorize(pr_vis), f"prediction (mIoU {score:.3f})")]):            axes[r, c].imshow(im); axes[r, c].set_title(t, fontsize=9); axes[r, c].axis("off")    handles = [Patch(color=PALETTE_RGB[i] / 255, label=n) for i, n in enumerate(cfg.CLASS_NAMES)]    handles.append(Patch(color="black", label="ignore/unannotated"))    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8, frameon=False)    fig.suptitle(f"{title} - {MODEL_TAG}, {cfg.SPLIT_MODE} fold {cfg.FOLD}", fontsize=12)    plt.tight_layout(rect=[0, 0.04, 1, 0.97])    p = f"{cfg.OUT_DIR}/qual_{title.replace(' ','_')}_{tag}.png"    plt.savefig(p, dpi=130, bbox_inches="tight"); plt.show()    hf_push(p, os.path.basename(p), "qualitative")triptych(best_imgs, "best cases")triptych(worst_imgs, "worst cases")

## 24. Confusion matrix figure

In [ ]:
cmn = np.array(final["confusion_matrix_norm"])fig, ax = plt.subplots(figsize=(7.2, 6))im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)ax.set_xticks(range(6)); ax.set_yticks(range(6))ax.set_xticklabels(cfg.CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(cfg.CLASS_NAMES)ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")ax.set_title(f"Row-normalized confusion - {MODEL_TAG}, {cfg.SPLIT_MODE} fold {cfg.FOLD}")for i in range(6):    for j in range(6):        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,                color="white" if cmn[i, j] > 0.5 else "black")fig.colorbar(im, fraction=0.046)plt.tight_layout()p = f"{cfg.OUT_DIR}/confusion_{tag}.png"plt.savefig(p, dpi=150, bbox_inches="tight"); plt.show()hf_push(p, os.path.basename(p), "confusion matrix")if WANDB: wandb.finish()

## 25. Standalone inference from Hugging FaceSelf-contained: point `INFER_REPO` at any fold's repo, give it images, get predictions. Nothing abovethis cell is required — it rebuilds the model from `best.pt`'s embedded config, so you can paste itinto a fresh notebook. `best.pt` holds the **EMA** weights, which is what the reported metrics refer to.Inputs are raw full-resolution tiles; the same `INTER_AREA` downsample as training is applied soinference matches training preprocessing exactly.This cell is longer than the U-Net equivalent for one reason: `smp.Unet` comes from a pip package, sothat notebook only had to import it. UNetFormer has no package, so the architecture is re-declaredhere. `DropPath` is inert at inference and carries no parameters, so the `state_dict` keys areidentical with or without it — but if you edit section 12, edit this too, or `load_state_dict` willfail loudly on key mismatch (which is the good outcome).

In [ ]:
# ============ STANDALONE: everything needed is inside this cell ============import os, numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F, timmfrom einops import rearrangefrom huggingface_hub import hf_hub_downloadclass Conv(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1, bias=False):        super().__init__(nn.Conv2d(i, o, k, stride, ((stride-1)+dilation*(k-1))//2, dilation, bias=bias))class ConvBN(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, o, k, stride, ((stride-1)+dilation*(k-1))//2, dilation, bias=False),                         nn.BatchNorm2d(o))class ConvBNReLU(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, o, k, stride, ((stride-1)+dilation*(k-1))//2, dilation, bias=False),                         nn.BatchNorm2d(o), nn.ReLU6(inplace=True))class SeparableConvBN(nn.Sequential):    def __init__(self, i, o, k=3, stride=1, dilation=1):        super().__init__(nn.Conv2d(i, i, k, stride, ((stride-1)+dilation*(k-1))//2, dilation,                                   groups=i, bias=False),                         nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o))class DropPath(nn.Module):    def __init__(self, p=0.0):        super().__init__(); self.p = float(p)    def forward(self, x):        if self.p == 0.0 or not self.training: return x        keep = 1.0 - self.p        return x * x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep) / keepclass Mlp(nn.Module):    def __init__(self, dim, hidden, drop=0.0):        super().__init__()        self.fc1, self.act = nn.Conv2d(dim, hidden, 1), nn.ReLU6(inplace=True)        self.fc2, self.drop = nn.Conv2d(hidden, dim, 1), nn.Dropout(drop)    def forward(self, x): return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))class GlobalLocalAttention(nn.Module):    def __init__(self, dim=64, num_heads=8, window_size=8, qkv_bias=False):        super().__init__()        self.nh, self.ws = num_heads, window_size        self.scale = (dim // num_heads) ** -0.5        self.qkv = Conv(dim, 3 * dim, k=1, bias=qkv_bias)        self.local1, self.local2 = ConvBN(dim, dim, k=3), ConvBN(dim, dim, k=1)        self.proj = SeparableConvBN(dim, dim, k=window_size)        self.attn_x = nn.AvgPool2d((window_size, 1), stride=1, padding=(window_size // 2 - 1, 0))        self.attn_y = nn.AvgPool2d((1, window_size), stride=1, padding=(0, window_size // 2 - 1))        ws = window_size        self.rpb_table = nn.Parameter(torch.zeros((2 * ws - 1) ** 2, num_heads))        c = torch.stack(torch.meshgrid(torch.arange(ws), torch.arange(ws), indexing="ij")).flatten(1)        rel = (c[:, :, None] - c[:, None, :]).permute(1, 2, 0).contiguous()        rel[:, :, 0] += ws - 1; rel[:, :, 1] += ws - 1; rel[:, :, 0] *= 2 * ws - 1        self.register_buffer("rpi", rel.sum(-1), persistent=False)    def forward(self, x):        B, C, H, W = x.shape        local = self.local2(x) + self.local1(x)        ws = self.ws        ph, pw = (ws - H % ws) % ws, (ws - W % ws) % ws        xp = F.pad(x, (0, pw, 0, ph), mode="reflect") if (ph or pw) else x        Hp, Wp = xp.shape[-2:]        q, k, v = rearrange(self.qkv(xp), "b (qkv h d) (hh w1) (ww w2) -> qkv (b hh ww) h (w1 w2) d",                            qkv=3, h=self.nh, w1=ws, w2=ws)        bias = self.rpb_table[self.rpi.view(-1)].view(ws * ws, ws * ws, self.nh).permute(2, 0, 1)        attn = ((q @ k.transpose(-2, -1)) * self.scale + bias.unsqueeze(0)).softmax(-1) @ v        attn = rearrange(attn, "(b hh ww) h (w1 w2) d -> b (h d) (hh w1) (ww w2)",                         b=B, hh=Hp // ws, ww=Wp // ws, w1=ws, w2=ws)[:, :, :H, :W]        fused = attn + local        cross = (self.attn_x(F.pad(fused, (0, 0, 0, 1), mode="reflect")) +                 self.attn_y(F.pad(fused, (0, 1, 0, 0), mode="reflect")))        return self.proj(F.pad(fused + cross, (0, 1, 0, 1), mode="reflect"))[:, :, :H, :W]class Block(nn.Module):    def __init__(self, dim=64, num_heads=8, window_size=8, mlp_ratio=4.0, drop=0.0, drop_path=0.0):        super().__init__()        self.norm1, self.norm2 = nn.BatchNorm2d(dim), nn.BatchNorm2d(dim)        self.attn = GlobalLocalAttention(dim, num_heads, window_size)        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)        self.dp1, self.dp2 = DropPath(drop_path), DropPath(drop_path)    def forward(self, x):        x = x + self.dp1(self.attn(self.norm1(x)))        return x + self.dp2(self.mlp(self.norm2(x)))class WF(nn.Module):    def __init__(self, in_ch, dc, eps=1e-8):        super().__init__()        self.pre_conv, self.post_conv = Conv(in_ch, dc, k=1), ConvBNReLU(dc, dc, k=3)        self.weights, self.eps = nn.Parameter(torch.ones(2, dtype=torch.float32)), eps    def forward(self, x, res):        x = F.interpolate(x, size=res.shape[-2:], mode="bilinear", align_corners=False)        w = torch.relu(self.weights); w = w / (w.sum() + self.eps)        return self.post_conv(w[0] * self.pre_conv(res) + w[1] * x)class FeatureRefinementHead(nn.Module):    def __init__(self, in_ch, dc, eps=1e-8):        super().__init__()        self.pre_conv, self.post_conv = Conv(in_ch, dc, k=1), ConvBNReLU(dc, dc, k=3)        self.weights, self.eps = nn.Parameter(torch.ones(2, dtype=torch.float32)), eps        self.pa = nn.Sequential(nn.Conv2d(dc, dc, 3, padding=1, groups=dc), nn.Sigmoid())        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), Conv(dc, max(dc // 16, 4), k=1), nn.ReLU6(True),                                Conv(max(dc // 16, 4), dc, k=1), nn.Sigmoid())        self.shortcut, self.proj = ConvBN(dc, dc, k=1), SeparableConvBN(dc, dc, k=3)        self.act = nn.ReLU6(True)    def forward(self, x, res):        x = F.interpolate(x, size=res.shape[-2:], mode="bilinear", align_corners=False)        w = torch.relu(self.weights); w = w / (w.sum() + self.eps)        x = self.post_conv(w[0] * self.pre_conv(res) + w[1] * x)        sc = self.shortcut(x)        x = self.pa(x) * x + self.ca(x) * x        return self.act(self.proj(x) + sc)class AuxHead(nn.Module):    def __init__(self, dc, n_cls):        super().__init__()        self.body = nn.Sequential(ConvBNReLU(dc, dc), nn.Dropout2d(0.1), Conv(dc, n_cls, k=1))    def forward(self, x): return self.body(x)class UNetFormer(nn.Module):    def __init__(self, encoder="resnet18", n_classes=6, dc=64, window_size=8,                 num_heads=8, dropout=0.1, drop_path=0.1, pretrained=True):        super().__init__()        self.backbone = timm.create_model(encoder, features_only=True,                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)        ec = self.backbone.feature_info.channels()        self.pre_conv = ConvBN(ec[3], dc, k=1)        self.b4 = Block(dc, num_heads, window_size, drop_path=drop_path)        self.p3, self.b3 = WF(ec[2], dc), Block(dc, num_heads, window_size, drop_path=drop_path)        self.p2, self.b2 = WF(ec[1], dc), Block(dc, num_heads, window_size, drop_path=drop_path)        self.p1 = FeatureRefinementHead(ec[0], dc)        self.aux_head = AuxHead(dc, n_classes)        self.seg_head = nn.Sequential(ConvBNReLU(dc, dc), nn.Dropout2d(dropout), Conv(dc, n_classes, k=1))    def forward(self, x):        h, w = x.shape[-2:]        r1, r2, r3, r4 = self.backbone(x)        d = self.b4(self.pre_conv(r4))        aux = self.aux_head(d)        d = self.b3(self.p3(d, r3))        d = self.b2(self.p2(d, r2))        d = self.p1(d, r1)        out = F.interpolate(self.seg_head(d), (h, w), mode="bilinear", align_corners=False)        if self.training:            return out, F.interpolate(aux, (h, w), mode="bilinear", align_corners=False)        return outdef load_from_hf(repo_id, token=None, device="cuda"):    ck = torch.load(hf_hub_download(repo_id, "best.pt", token=token),                    map_location=device, weights_only=False)    c = ck["cfg"]    m = UNetFormer(encoder=c["ENCODER"], n_classes=c["N_CLASSES"], dc=c["DECODE_CHANNELS"],                   window_size=c["WINDOW_SIZE"], num_heads=c["NUM_HEADS"], dropout=c["DROPOUT"],                   drop_path=c.get("DROP_PATH", 0.0),                   pretrained=False).to(device).eval()      # weights come from the checkpoint    m.load_state_dict(ck["model_state"])    return m, c, ck["metrics"]@torch.no_grad()def predict_paths(model, c, paths, device="cuda", batch=8):    mean = np.array([0.485, 0.456, 0.406], np.float32); std = np.array([0.229, 0.224, 0.225], np.float32)    out = []    for i in range(0, len(paths), batch):        xs = []        for p in paths[i:i + batch]:            im = cv2.imread(str(p), cv2.IMREAD_COLOR)            im = cv2.resize(im, (c["IMG_SIZE"], c["IMG_SIZE"]), interpolation=cv2.INTER_AREA)            a = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0            xs.append(torch.from_numpy(((a - mean) / std).transpose(2, 0, 1)))        x = torch.stack(xs).to(device)        with torch.amp.autocast("cuda"):            out.append(model(x).float().argmax(1).cpu().numpy())    return np.concatenate(out)# --- example ---INFER_REPO = f"{cfg.HF_USER}/{cfg.HF_PREFIX}-block-fold0"inf_model, inf_cfg, inf_metrics = load_from_hf(INFER_REPO, token=HF_TOKEN)print(f"loaded {INFER_REPO} | val mIoU {inf_metrics['mIoU']:.4f}")demo = [IMG_DIR / r[2] for r in va_recs[:4]]preds = predict_paths(inf_model, inf_cfg, demo)print("pred shape:", preds.shape, "| classes present:", np.unique(preds).tolist())

## Run checklist1. Internet **ON**; Secrets `HF_TOKEN` (write) + `WANDB_API_KEY`; set `cfg.HF_USER`.2. Section 5 must print **channel-order check PASSED**.3. Section 10 must print **augmentation varies across epochs: True**. If it prints `False`, someone   set `persistent_workers=True` on the train loader again — that silently disables on-the-fly   augmentation for the whole run.4. Section 12 must print **~11.8 M params**, the arch signature, `EMA: on`, and an LR schedule line   whose value at epoch 120 is `0.00e+00`. If it is not zero at 120, the cosine horizon is wrong.5. The 512 cache is model-independent. If you already built it for another baseline, set   `PREBUILT_CACHE` and section 6 is a no-op; otherwise let it repack once (~15-20 min), then   `Save Version`, attach the output as a dataset, and set it from then on.6. Run six fold-runs, one per session, changing only `SPLIT_MODE` and `FOLD`:   `block/0, block/1, block/2, random/0, random/1, random/2`.7. Re-running the same fold after a timeout resumes automatically. If you deliberately change the   architecture, bump `ARCH_VERSION` — the resume guard will then discard the old checkpoints for you   instead of half-loading them.8. After all six, run section 22 for the aggregate table and the leakage gap.### What to watch in the first ten epochs- **Gradient norms.** Single digits after warmup is healthy. `grad_norm_max` in the hundreds and  climbing means drop `LR_DECODER` to 1.5e-4 and record the deviation.- **EMA vs raw mIoU.** EMA should track at or above raw within a few epochs. Consistently below means  the decay is too slow for this run length; drop `EMA_DECAY` to 0.995.- **The LR curve in section 21.** It must reach zero at epoch 120. This was the single largest  inconsistency in the previous version of this notebook.### If it still overfitsThe levers, in the order to try them, applied to **all five models** and not just this one:`DROP_PATH` 0.1 -> 0.2, then `WEIGHT_DECAY` 0.01 -> 0.05, then `COS_HORIZON_EPOCHS` 120 -> 80. Do notlower `PATIENCE` first — with `MIN_DELTA` in place, patience is no longer what keeps a plateaued runalive.### When you write this upReport `lr = 3e-4` as a deliberate reduction from the published `6e-4` for cross-baseline parity atthis dataset scale; call the block protocol *sequence-block CV (export-order proxy)*; give theleakage gap as a lower bound; state that reported weights are EMA; note that reproducibility is exactexcept for non-deterministic GPU reduction order; and note that the UNetFormer implementation isindependent of the GPL-3.0 reference repository.